# 用 Jupyter + LLM 学懂梯度下降

这个 Notebook 不追求把代码写得多复杂，而是走完一次 **问、猜、跑、看、讲、测** 的学习闭环。

学习目标：

1. 解释梯度下降公式中每个符号的含义；
2. 观察单调收敛、振荡收敛和发散；
3. 从具体函数推导学习率的收敛范围；
4. 可选地调用 OpenAI-compatible LLM API，让 AI 充当追问型学习教练。

> 安全提醒：不要把 API key 直接写进 Notebook。代码只从环境变量读取密钥，默认也不会发送任何 API 请求。不要把公司机密、个人隐私、未公开代码或无权交给第三方处理的材料放进提示词。

## 0. 准备环境

如果环境里缺少依赖，可以在一个新 Cell 中运行：

```python
%pip install numpy matplotlib openai
```

调用 LLM 前，在启动 Jupyter 的终端中配置环境变量：

```bash
export LLM_API_KEY=your-api-key
export LLM_BASE_URL=https://your-openai-compatible-endpoint/v1
export LLM_MODEL=your-model-name
```

也可以使用 `OPENAI_API_KEY` 和 `OPENAI_BASE_URL`。不要在 Notebook 中打印密钥。

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.figsize"] = (12, 6)

## 1. 先看懂公式

梯度下降的更新公式是：

$$
x_{t+1}=x_t-\eta\nabla f(x_t)
$$

- $f(x)$：想要最小化的目标函数或损失函数；
- $x_t$：第 $t$ 步的当前参数；
- $x_{t+1}$：更新后的参数；
- $\nabla f(x_t)$：当前位置的梯度，指向函数上升最快的方向；
- $\eta$：学习率，控制每次迈多大一步；
- 负号：让参数沿梯度的反方向，也就是下坡方向移动。

本实验使用：

$$
f(x)=(x-3)^2+2,\qquad \nabla f(x)=2(x-3)
$$

## 2. 先猜，再运行

请先填写，不要急着执行后面的图：

1. 最低点大约在 $x=$ ______。
2. $\eta=0.1$ 时，轨迹会 ______。
3. $\eta=0.8$ 时，轨迹会 ______。
4. $\eta=1.05$ 时，轨迹会 ______。
5. 我目前的信心：______ / 5。

In [ ]:
def f(x):
    return (x - 3) ** 2 + 2


def grad(x):
    return 2 * (x - 3)


def gradient_descent(x0, eta, steps=20):
    xs = [float(x0)]
    for _ in range(steps):
        xs.append(xs[-1] - eta * grad(xs[-1]))
    return np.array(xs)

In [ ]:
x0 = -6.0
eta = 0.1
gradient = grad(x0)
x1 = x0 - eta * gradient

print(f"x0 = {x0}")
print(f"gradient = {gradient}")
print(f"x1 = {x1}")
assert np.isclose(x1, -4.2)

In [ ]:
rates = [0.1, 0.8, 1.05]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col, eta in enumerate(rates):
    xs = gradient_descent(x0=-6, eta=eta)
    left = min(-8, xs.min() - 1)
    right = max(12, xs.max() + 1)
    grid = np.linspace(left, right, 400)

    path_ax = axes[0, col]
    path_ax.plot(grid, f(grid), color="steelblue")
    path_ax.scatter(xs, f(xs), c=np.arange(len(xs)), cmap="autumn", s=28)
    path_ax.plot(xs, f(xs), color="gray", alpha=0.5)
    path_ax.set_title(f"eta = {eta}")
    path_ax.set_xlabel("x")
    path_ax.grid(alpha=0.25)

    error_ax = axes[1, col]
    error_ax.plot(np.arange(len(xs)), np.abs(xs - 3), marker="o")
    error_ax.set_yscale("log")
    error_ax.set_xlabel("iteration")
    error_ax.grid(alpha=0.25)

axes[0, 0].set_ylabel("f(x)")
axes[1, 0].set_ylabel("|x - 3|")
plt.tight_layout()
plt.show()

assert abs(gradient_descent(-6, 0.1)[-1] - 3) < 0.11
assert abs(gradient_descent(-6, 0.8)[-1] - 3) < 0.001
assert abs(gradient_descent(-6, 1.05)[-1] - 3) > 9

## 3. 记录观察和解释

先写观察，再写解释，不要把两者混在一起。

- **观察**：$\eta=0.1$ 时，我看到 ______。
- **解释**：因为每轮误差会 ______。
- **观察**：$\eta=0.8$ 时，我看到 ______。
- **解释**：负号意味着 ______，绝对值小于 1 意味着 ______。
- **观察**：$\eta=1.05$ 时，我看到 ______。
- **解释**：它越跑越远，是因为 ______。

对这个特定函数：

$$
x_{t+1}-3=(1-2\eta)(x_t-3)
$$

要让误差逐步缩小，需要 $|1-2\eta|<1$，因此 $0<\eta<1$。这个范围只属于当前函数和当前假设，不是梯度下降的万能学习率。

## 4. 可选：让 LLM 充当学习陪练

下面使用兼容 OpenAI Chat Completions 的 API。为了避免误调用，`ENABLE_LLM_CALL` 默认为 `False`。

启用前请确认：

- API 地址、模型名称和计费规则正确；
- 提示词中没有秘密、个人数据或受限制材料；
- 你允许这些内容被发送给对应的服务提供者。

In [ ]:
ENABLE_LLM_CALL = False  # 确认配置和数据边界后，手动改成 True

api_key = os.getenv("LLM_API_KEY") or os.getenv("OPENAI_API_KEY")
base_url = os.getenv("LLM_BASE_URL") or os.getenv("OPENAI_BASE_URL")
model = os.getenv("LLM_MODEL", "gpt-4o-mini")
client = None

if ENABLE_LLM_CALL:
    if not api_key:
        raise RuntimeError("请先设置 LLM_API_KEY 或 OPENAI_API_KEY")

    from openai import OpenAI

    client_options = {"api_key": api_key, "timeout": 30.0}
    if base_url:
        client_options["base_url"] = base_url
    client = OpenAI(**client_options)
    print(f"LLM API enabled; model={model}")
else:
    print("LLM API disabled; set ENABLE_LLM_CALL=True to enable it.")

In [ ]:
def ask_learning_coach(question, my_observation):
    if client is None:
        return "LLM API 尚未启用。请先检查配置，再显式打开 ENABLE_LLM_CALL。"

    system_prompt = """
你是梯度下降学习陪练，不是代答工具。
规则：
1. 先判断学习者的观察是否具体；
2. 一次只追问一个关键问题；
3. 不直接给完整答案，优先给最小提示；
4. 要求学习者把图像现象和公式中的误差倍率联系起来；
5. 如果发现错误，明确指出错误发生在哪一步；
6. 回复控制在 200 个汉字以内。
""".strip()

    user_prompt = f"问题：{question}\n我的观察：{my_observation}"
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
        max_tokens=400,
    )
    return response.choices[0].message.content or ""

In [ ]:
question = "为什么 eta=0.8 会左右振荡，却仍然收敛？"
my_observation = (
    "x 在最低点 3 的两边交替出现，"
    "下排误差曲线持续下降，但我还说不清它与 1-2*eta 的关系。"
)

print(ask_learning_coach(question, my_observation))

## 5. 闭卷验收

关掉上面的答案，考虑新函数：

$$
g(x)=4(x+2)^2+1
$$

请先手算，再写代码验证：

1. 最低点在哪里？
2. 梯度是什么？
3. 固定学习率满足什么范围时，误差会逐步缩小？
4. 从最低点右侧出发，第一次更新往哪个方向走？
5. 设计三个学习率，分别演示单调收敛、振荡收敛和发散。

如果能解释边界、预测轨迹并用实验验证，这个公式才算真正开始属于你。